# Session 5 — interactive charts + mini-project

The charts so far are static images. **plotly express** makes **interactive** charts: hover for details, zoom, hide groups, even animate over time.
Then you put everything together in a mini-project.

**You will learn to:**
- make interactive scatter, line and map charts with plotly express
- animate a chart over time and save it as an HTML file
- answer a question with data on your own, from loading to conclusion

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

sns.set_theme(style="whitegrid")
gm = px.data.gapminder()
gm07 = gm[gm["year"] == 2007]

## Part 1 — plotly express

The pattern is almost the same as seaborn: `px.xxx(df, x=, y=, color=, size=)`.
`hover_name` sets what shows when you hover over a dot, `log_x=True` gives a log scale.
Try it: hover over the dots, drag to zoom (double-click to reset), click a continent in the legend to hide it.

In [ ]:
fig = px.scatter(gm07, x="gdpPercap", y="lifeExp", color="continent",
                 size="pop", hover_name="country", log_x=True, size_max=55,
                 title="Wealth vs. health, 2007")
fig.show()

Add `animation_frame="year"` and the chart gets a **play button** — this is the famous Hans Rosling chart.
`animation_group` tells plotly which dot is the same country across years.
`range_x` / `range_y` fix the axes so they don't jump between years.

In [ ]:
fig = px.scatter(gm, x="gdpPercap", y="lifeExp", color="continent",
                 size="pop", hover_name="country", log_x=True, size_max=55,
                 animation_frame="year", animation_group="country",
                 range_x=[100, 100_000], range_y=[25, 90],
                 title="50 years of wealth and health — Gapminder")
fig.show()

Interactive line chart: with many lines, hover and legend clicks make it readable.

In [ ]:
fig = px.line(gm[gm["continent"] == "Asia"], x="year", y="lifeExp",
              color="country", title="Life expectancy in Asia")
fig.show()

A **choropleth** colors countries on a world map. `locations` needs country codes (`iso_alpha` = 3-letter codes like `IRN`).
`range_color` keeps the color scale the same for every year.

In [ ]:
fig = px.choropleth(gm, locations="iso_alpha", color="lifeExp", hover_name="country",
                    animation_frame="year", color_continuous_scale="Viridis",
                    range_color=[25, 85], title="Life expectancy around the world")
fig.show()

`write_html` saves an interactive chart as a standalone file — it opens in any browser and you can send it to anyone.

In [ ]:
fig.write_html("life_expectancy_map.html")

## Part 2 — mini-project

**Answer one question with data.** Use Titanic, Gapminder, or another built-in dataset:
`sns.load_dataset("tips")`, `sns.load_dataset("penguins")`, `px.data.stocks()`.

Steps:
1. Write your question in a markdown cell at the top.
2. Load → look (`head`, `info`, `describe`) → clean (missing values?).
3. Filter / `groupby` to get the numbers that answer the question.
4. Make 3–4 charts. Each has a title, axis labels, and **one sentence takeaway** below it.
5. Final cell: your answer in 2–3 sentences.

Question ideas:
- Has Asia caught up with Europe in life expectancy?
- Do penguin species differ in size? Does sex matter?
- Who tips more — smokers or non-smokers, lunch or dinner?
- Which tech stock grew most (`px.data.stocks()`)?

Below is a complete example project. If you get stuck, look at how it is structured — but try your own question first.

### Example mini-project — *Has Asia caught up with Europe in life expectancy?*

In [ ]:
# 1-2. load & look — Gapminder is already clean
asia_europe = gm[gm["continent"].isin(["Asia", "Europe"])]
asia_europe.groupby("continent")["country"].nunique()

In [ ]:
# 3. average per continent per year, and the gap
avg = asia_europe.groupby(["year", "continent"])["lifeExp"].mean().unstack()
avg["gap"] = avg["Europe"] - avg["Asia"]
avg.round(1)

In [ ]:
# 4a. the two lines
ax = sns.lineplot(data=asia_europe, x="year", y="lifeExp", hue="continent", errorbar=None, marker="o")
ax.set_title("Average life expectancy: Asia vs. Europe")
ax.set_ylabel("Life expectancy (years)")
plt.show()

*Takeaway: both rise, but Asia rises faster.*

In [ ]:
# 4b. the gap itself
ax = avg["gap"].plot(kind="bar", color="tab:gray")
ax.set_title("How many years Europeans outlive Asians, on average")
ax.set_xlabel("")
ax.set_ylabel("Gap (years)")
plt.show()

*Takeaway: the gap shrank by more than half between 1952 and 2007.*

In [ ]:
# 4c. spread inside each continent in 2007 — averages hide a lot
ax = sns.boxplot(data=asia_europe[asia_europe["year"] == 2007], x="continent", y="lifeExp")
ax.set_title("Life expectancy across countries, 2007")
plt.show()

*Takeaway: Asia's countries are much more spread out — some match Europe, others are far behind.*

**Answer:** Not fully. Asia closed most of the gap since 1952, but the average still trails Europe
by several years in 2007, and the variation between Asian countries is large.

## Where to go next
- **pandas:** "10 minutes to pandas" in the official docs; then learn `merge` (joining tables), `pivot_table`, and dates (`pd.to_datetime`).
- **Charts:** the seaborn gallery and plotly express gallery — find a chart you like, copy it, change it.
- **Practice:** Kaggle Learn (free *Pandas* and *Data Visualization* courses), then pick any Kaggle dataset you find interesting.
- **Next level:** Streamlit, to turn a notebook into a small web dashboard.

The only way it sticks: **one small question per week, with your own data.**